# AVISTA Validation Study — UCI Mushroom Dataset

## Objective

This notebook validates the AVISTA machine-learning workflow using the public **UCI Mushroom** dataset. The primary objective is to determine whether AVISTA can reproduce results obtained from a direct Jupyter implementation under matched experimental conditions, while also comparing the workflow with AutoGluon as an external AutoML platform.

## Dataset

The UCI Mushroom dataset contains **8,124 observations** describing mushrooms using categorical physical characteristics. The prediction target is binary:

* `e` = edible
* `p` = poisonous

The original dataset contains 22 predictors. The constant `veil-type` feature is removed because it contains only one observed category. Missing categorical values, primarily in `stalk-root`, are retained in the raw dataset and handled during preprocessing.

## Experimental Setup

A fixed random seed of **42** is used throughout the experiment. The dataset is divided using stratified sampling into:

* **70% training**
* **10% validation**
* **20% test**

For the direct Jupyter and AVISTA experiments, preprocessing is fitted using training data only. Missing categorical values are represented by the reserved category `Unknown`, followed by one-hot encoding. Any imbalance handling is applied exclusively to the training data. The selected AVISTA setting is **SMOTE Moderate (0.5)**; however, no synthetic samples are generated when the existing minority-class count already exceeds the specified target ratio.

Five-fold stratified cross-validation is performed within the training partition using leakage-safe preprocessing, where preprocessing artifacts are independently fitted within each fold.

The evaluated models are:

* Logistic Regression
* Random Forest
* XGBoost

The primary evaluation metrics are **Accuracy, Macro-F1, Precision, Recall, and ROC-AUC**. Training/effective sample size and runtime are also recorded.

## AutoGluon Comparison

AutoGluon is evaluated using the same raw train, validation, and test observations. AutoGluon is allowed to use its native preprocessing and 5-fold bagging workflow rather than reproducing AVISTA's internal preprocessing exactly. Therefore, the AutoGluon experiment is treated as a **platform-level comparison**, while AVISTA versus direct Jupyter represents the controlled reproducibility comparison.

## Additional Analysis

Because all predictors are categorical, **Cramér's V** is used instead of Pearson correlation to examine categorical associations. Pairwise feature associations are also inspected to identify highly redundant predictors before final model evaluation.


In [ ]:
!pip install ucimlrepo

In [29]:
# ============================================================
# Cell 2: Download and load the UCI Mushroom dataset
# ============================================================

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Download Mushroom dataset from UCI
mushroom = fetch_ucirepo(id=73)

# Separate features and target
X = mushroom.data.features.copy()
y = mushroom.data.targets.copy()

# Combine into one dataframe for inspection
df = pd.concat([X, y], axis=1)

print("Dataset loaded successfully.")
print(f"Total rows     : {df.shape[0]}")
print(f"Features       : {X.shape[1]}")
print(f"Target column  : {y.columns[0]}")
print(f"Dataset shape  : {df.shape}")

display(df.head())

Dataset loaded successfully.
Total rows     : 8124
Features       : 22
Target column  : poisonous
Dataset shape  : (8124, 23)


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e


In [30]:
# ============================================================
# Cell 3: Save raw Mushroom dataset for AVISTA
# ============================================================

from pathlib import Path

# Create output folder
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

# Save features + target exactly as loaded from UCI
raw_csv_path = data_dir / "uci_mushroom_raw.csv"
df.to_csv(raw_csv_path, index=False)

print(f"Saved raw dataset to: {raw_csv_path}")
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

# Verify the saved file
df_check = pd.read_csv(raw_csv_path)

print("\nSaved-file verification:")
print(f"Shape   : {df_check.shape}")
print(f"Target  : {'poisonous' in df_check.columns}")

display(df_check.head())

Saved raw dataset to: data\uci_mushroom_raw.csv
Rows    : 8124
Columns : 23

Saved-file verification:
Shape   : (8124, 23)
Target  : True


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,w,w,p,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,w,w,p,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,w,w,p,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,w,w,p,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,w,w,p,w,o,e,n,a,g,e


In [31]:
# ============================================================
# Cell 4: Raw data integrity check
# ============================================================

print("=== DATASET SHAPE ===")
print(f"Rows     : {df.shape[0]}")
print(f"Features : {X.shape[1]}")
print(f"Target   : {y.columns[0]}")

print("\n=== TARGET DISTRIBUTION ===")
target_counts = df["poisonous"].value_counts()
target_pct = df["poisonous"].value_counts(normalize=True) * 100

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percent": target_pct.round(2)
})

display(target_summary)

print("\n=== MISSING VALUES ===")
missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Percent": (df.isna().mean() * 100).round(2)
})

display(missing_summary[missing_summary["Missing"] > 0])

print("\n=== DUPLICATE ROWS ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== DATA TYPES ===")
display(df.dtypes.to_frame("dtype"))

print("\n=== UNIQUE VALUES PER FEATURE ===")
unique_summary = pd.DataFrame({
    "Unique values": X.nunique(dropna=False)
}).sort_values("Unique values")

display(unique_summary)

=== DATASET SHAPE ===
Rows     : 8124
Features : 22
Target   : poisonous

=== TARGET DISTRIBUTION ===


,Count,Percent
poisonous,,
e,4208,51.8
p,3916,48.2



=== MISSING VALUES ===


,Missing,Percent
stalk-root,2480,30.53



=== DUPLICATE ROWS ===
Duplicate rows: 0

=== DATA TYPES ===


,dtype
cap-shape,object
cap-surface,object
cap-color,object
bruises,object
odor,object
gill-attachment,object
gill-spacing,object
gill-size,object
gill-color,object
stalk-shape,object



=== UNIQUE VALUES PER FEATURE ===


,Unique values
veil-type,1
bruises,2
gill-spacing,2
gill-size,2
stalk-shape,2
gill-attachment,2
ring-number,3
cap-surface,4
stalk-surface-above-ring,4
stalk-surface-below-ring,4


In [32]:
# ============================================================
# Cell 5: Drop constant feature and save AVISTA dataset
# ============================================================

from pathlib import Path

# Drop feature with only one unique value
DROP_COLUMNS = ["veil-type"]

df_avista = df.drop(columns=DROP_COLUMNS).copy()

print("Dropped columns:", DROP_COLUMNS)
print(f"Updated dataset shape : {df_avista.shape}")
print(f"Features              : {df_avista.shape[1] - 1}")
print(f"Target                : poisonous")

# ------------------------------------------------------------
# Save updated raw dataset for AVISTA
# Missing categorical values are intentionally preserved here
# ------------------------------------------------------------

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

avista_csv_path = data_dir / "uci_mushroom_avista.csv"

df_avista.to_csv(
    avista_csv_path,
    index=False
)

print(f"\nSaved AVISTA dataset to: {avista_csv_path}")

# Verify saved file
df_check = pd.read_csv(avista_csv_path)

print(f"Saved shape : {df_check.shape}")
print(f"Missing stalk-root values : {df_check['stalk-root'].isna().sum()}")

display(df_check.head())

Dropped columns: ['veil-type']
Updated dataset shape : (8124, 22)
Features              : 21
Target                : poisonous

Saved AVISTA dataset to: data\uci_mushroom_avista.csv
Saved shape : (8124, 22)
Missing stalk-root values : 2480


,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,stalk-shape,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-color,ring-number,ring-type,spore-print-color,population,habitat,poisonous
0,x,s,n,t,p,f,c,n,k,e,...,s,w,w,w,o,p,k,s,u,p
1,x,s,y,t,a,f,c,b,k,e,...,s,w,w,w,o,p,n,n,g,e
2,b,s,w,t,l,f,c,b,n,e,...,s,w,w,w,o,p,n,n,m,e
3,x,y,w,t,p,f,c,n,n,e,...,s,w,w,w,o,p,k,s,u,p
4,x,s,g,f,n,f,w,b,k,t,...,s,w,w,w,o,e,n,a,g,e


In [44]:
# ============================================================
# Cell 5B: Remove highly associated categorical features
# Pairwise Cramér's V threshold = 0.80
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

CRAMERS_V_THRESHOLD = 0.65
TARGET = "poisonous"

# ------------------------------------------------------------
# Cramér's V function
# ------------------------------------------------------------


def cramers_v(x, y):
    x = x.fillna("Unknown").astype(str)
    y = y.fillna("Unknown").astype(str)

    table = pd.crosstab(x, y)

    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()

    if n == 0:
        return 0.0

    r, k = table.shape

    denominator = min(k - 1, r - 1)

    if denominator == 0:
        return 0.0

    return np.sqrt((chi2 / n) / denominator)


# ------------------------------------------------------------
# Features only — TARGET IS NOT USED
# ------------------------------------------------------------
X_corr = df_avista.drop(columns=TARGET).copy()
features = X_corr.columns.tolist()

# Pairwise Cramér's V matrix
cramers_matrix = pd.DataFrame(
    np.eye(len(features)),
    index=features,
    columns=features
)

for i in range(len(features)):
    for j in range(i + 1, len(features)):

        v = cramers_v(
            X_corr[features[i]],
            X_corr[features[j]]
        )

        cramers_matrix.iloc[i, j] = v
        cramers_matrix.iloc[j, i] = v


# ------------------------------------------------------------
# Find highly associated feature pairs
# ------------------------------------------------------------
high_pairs = []

for i in range(len(features)):
    for j in range(i + 1, len(features)):

        v = cramers_matrix.iloc[i, j]

        if v >= CRAMERS_V_THRESHOLD:
            high_pairs.append({
                "Feature 1": features[i],
                "Feature 2": features[j],
                "Cramer's V": v
            })

high_pairs_df = pd.DataFrame(high_pairs)

print(
    f"=== HIGHLY ASSOCIATED FEATURE PAIRS "
    f"(V >= {CRAMERS_V_THRESHOLD}) ==="
)

if len(high_pairs_df) > 0:
    display(
        high_pairs_df.sort_values(
            "Cramer's V",
            ascending=False
        ).style.format({
            "Cramer's V": "{:.4f}"
        })
    )
else:
    print("No highly associated feature pairs found.")


# ------------------------------------------------------------
# Drop redundant features
#
# For each highly associated pair, drop the feature having
# the higher average association with all other predictors.
# This retains the comparatively less redundant feature.
# ------------------------------------------------------------
features_to_drop = set()

for pair in high_pairs:

    f1 = pair["Feature 1"]
    f2 = pair["Feature 2"]

    # Skip if one has already been removed
    if f1 in features_to_drop or f2 in features_to_drop:
        continue

    mean_v_f1 = cramers_matrix.loc[
        f1,
        cramers_matrix.columns != f1
    ].mean()

    mean_v_f2 = cramers_matrix.loc[
        f2,
        cramers_matrix.columns != f2
    ].mean()

    if mean_v_f1 > mean_v_f2:
        features_to_drop.add(f1)
    else:
        features_to_drop.add(f2)


features_to_drop = sorted(features_to_drop)

print("\nFeatures to drop:")
print(features_to_drop)

# ------------------------------------------------------------
# Update dataset used by Cell 6 onward
# ------------------------------------------------------------
df_avista = df_avista.drop(
    columns=features_to_drop
).copy()

print("\n=== REDUCED DATASET ===")
print(f"Rows               : {df_avista.shape[0]}")
print(f"Remaining features : {df_avista.shape[1] - 1}")
print(f"Target             : {TARGET}")
print(f"Dropped features   : {len(features_to_drop)}")

=== HIGHLY ASSOCIATED FEATURE PAIRS (V >= 0.65) ===


,Feature 1,Feature 2,Cramer's V
7,gill-size,gill-color,0.7841
2,bruises,ring-type,0.7685
3,odor,gill-size,0.7609
4,odor,ring-number,0.7428
9,ring-number,ring-type,0.7139
5,odor,ring-type,0.6969
6,gill-spacing,population,0.6739
0,bruises,odor,0.6609
8,gill-size,spore-print-color,0.6607
1,bruises,gill-color,0.6503



Features to drop:
['gill-color', 'odor', 'population', 'ring-type', 'spore-print-color']

=== REDUCED DATASET ===
Rows               : 8124
Remaining features : 13
Target             : poisonous
Dropped features   : 5


In [45]:
# ============================================================
# Cell 5C: Save reduced Mushroom dataset for AVISTA
# ============================================================

from pathlib import Path
import pandas as pd

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

avista_reduced_path = data_dir / "uci_mushroom_avista_reduced.csv"

# Save reduced dataset
df_avista.to_csv(
    avista_reduced_path,
    index=False
)

# Verify saved file
df_check = pd.read_csv(avista_reduced_path)

print(f"Saved to          : {avista_reduced_path}")
print(f"Rows              : {df_check.shape[0]}")
print(f"Features          : {df_check.shape[1] - 1}")
print(f"Total columns     : {df_check.shape[1]}")
print(f"Target            : poisonous")
print(f"Missing stalk-root: {df_check['stalk-root'].isna().sum()}")

display(df_check.head())

Saved to          : data\uci_mushroom_avista_reduced.csv
Rows              : 8124
Features          : 13
Total columns     : 14
Target            : poisonous
Missing stalk-root: 2480


,cap-shape,cap-surface,cap-color,bruises,gill-spacing,gill-size,stalk-shape,stalk-root,stalk-surface-above-ring,stalk-surface-below-ring,veil-color,ring-number,habitat,poisonous
0,x,s,n,t,c,n,e,e,s,s,w,o,u,p
1,x,s,y,t,c,b,e,c,s,s,w,o,g,e
2,b,s,w,t,c,b,e,c,s,s,w,o,m,e
3,x,y,w,t,c,n,e,e,s,s,w,o,u,p
4,x,s,g,f,w,b,t,e,s,s,w,o,g,e


In [46]:
# ============================================================
# Cell 6: Stratified 70/10/20 train-validation-test split
# ============================================================

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TARGET = "poisonous"

# Separate features and target
X = df_avista.drop(columns=TARGET)
y = df_avista[TARGET]

# ------------------------------------------------------------
# Step 1: 70% training, 30% temporary
# ------------------------------------------------------------
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y
)

# ------------------------------------------------------------
# Step 2: Split temporary data into
# 10% validation and 20% test
#
# Validation = 1/3 of temporary 30%
# Test       = 2/3 of temporary 30%
# ------------------------------------------------------------
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    random_state=RANDOM_SEED,
    stratify=y_temp
)

print("=== DATA SPLIT ===")
print(f"Random seed : {RANDOM_SEED}")
print(f"Training    : {len(X_train)} ({len(X_train)/len(df_avista)*100:.2f}%)")
print(f"Validation  : {len(X_val)} ({len(X_val)/len(df_avista)*100:.2f}%)")
print(f"Test        : {len(X_test)} ({len(X_test)/len(df_avista)*100:.2f}%)")

print("\n=== CLASS DISTRIBUTION ===")

for name, target in [
    ("Training", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print(target.value_counts())
    print((target.value_counts(normalize=True) * 100).round(2))

=== DATA SPLIT ===
Random seed : 42
Training    : 5686 (69.99%)
Validation  : 812 (10.00%)
Test        : 1626 (20.01%)

=== CLASS DISTRIBUTION ===

Training
poisonous
e    2945
p    2741
Name: count, dtype: int64
poisonous
e    51.79
p    48.21
Name: proportion, dtype: float64

Validation
poisonous
e    421
p    391
Name: count, dtype: int64
poisonous
e    51.85
p    48.15
Name: proportion, dtype: float64

Test
poisonous
e    842
p    784
Name: count, dtype: int64
poisonous
e    51.78
p    48.22
Name: proportion, dtype: float64


In [47]:
# ============================================================
# Cell 7: AVISTA-equivalent preprocessing
# Fit on TRAIN only
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Identify feature types
categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

print(f"Categorical features : {len(categorical_features)}")
print(f"Numerical features   : {len(numerical_features)}")

# ------------------------------------------------------------
# Categorical preprocessing:
# missing -> "Unknown"
# then one-hot encode
# ------------------------------------------------------------
categorical_pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="constant",
            fill_value="Unknown"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

# Mushroom has no numerical features, so only categorical
# preprocessing is required here.
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

# ------------------------------------------------------------
# IMPORTANT: fit preprocessing on TRAIN only
# ------------------------------------------------------------
X_train_processed = preprocessor.fit_transform(X_train)

# Validation/test use the TRAIN-fitted preprocessing
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("\n=== PROCESSED DATA ===")
print("Train      :", X_train_processed.shape)
print("Validation :", X_val_processed.shape)
print("Test       :", X_test_processed.shape)

# Verify the preprocessing artifact was fitted only once
print("\nPreprocessing fitted on training data only.")

Categorical features : 13
Numerical features   : 0

=== PROCESSED DATA ===
Train      : (5686, 55)
Validation : (812, 55)
Test       : (1626, 55)

Preprocessing fitted on training data only.


In [48]:
# ============================================================
# Cell 8: AVISTA-equivalent SMOTE Moderate (0.5)
# Apply to TRAIN only
# ============================================================

import numpy as np
from imblearn.over_sampling import SMOTE

SMOTE_RATIO = 0.5
SMOTE_SEED = 42

# Class counts before SMOTE
class_counts = y_train.value_counts()

majority_class = class_counts.idxmax()
minority_class = class_counts.idxmin()

majority_count = class_counts[majority_class]
minority_count = class_counts[minority_class]

# AVISTA Moderate rule
target_minority_count = max(
    minority_count,
    int(np.floor(SMOTE_RATIO * majority_count))
)

print("=== BEFORE SMOTE ===")
print(class_counts)
print(f"\nMajority class          : {majority_class}")
print(f"Majority count          : {majority_count}")
print(f"Minority class          : {minority_class}")
print(f"Current minority count  : {minority_count}")
print(f"Target minority count   : {target_minority_count}")

# Apply SMOTE only when additional minority samples are required
if target_minority_count > minority_count:

    smote = SMOTE(
        sampling_strategy={minority_class: target_minority_count},
        random_state=SMOTE_SEED
    )

    X_train_final, y_train_final = smote.fit_resample(
        X_train_processed,
        y_train
    )

    print("\nSMOTE applied.")

else:
    X_train_final = X_train_processed
    y_train_final = y_train.copy()

    print("\nSMOTE not required: minority class already exceeds the 0.5 target.")

print("\n=== AFTER SMOTE ===")
print(y_train_final.value_counts())

print(f"\nOriginal training N : {len(y_train)}")
print(f"Effective training N: {len(y_train_final)}")

=== BEFORE SMOTE ===
poisonous
e    2945
p    2741
Name: count, dtype: int64

Majority class          : e
Majority count          : 2945
Minority class          : p
Current minority count  : 2741
Target minority count   : 2741

SMOTE not required: minority class already exceeds the 0.5 target.

=== AFTER SMOTE ===
poisonous
e    2945
p    2741
Name: count, dtype: int64

Original training N : 5686
Effective training N: 5686


In [49]:
# ============================================================
# Cell 9: 5-Fold CV — Logistic Regression
# Leakage-safe AVISTA-equivalent pipeline
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

from imblearn.over_sampling import SMOTE

RANDOM_SEED = 42
N_FOLDS = 5
SMOTE_RATIO = 0.5

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_SEED
)

cv_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):

    # --------------------------------------------------------
    # Fold train / fold validation
    # --------------------------------------------------------
    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()

    X_fold_val = X_train.iloc[val_idx].copy()
    y_fold_val = y_train.iloc[val_idx].copy()

    # --------------------------------------------------------
    # Fit preprocessing on FOLD TRAIN only
    # --------------------------------------------------------
    categorical_features = X_fold_train.columns.tolist()

    categorical_pipeline = Pipeline([
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Unknown"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ])

    fold_preprocessor = ColumnTransformer([
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ])

    X_fold_train_processed = fold_preprocessor.fit_transform(
        X_fold_train
    )

    X_fold_val_processed = fold_preprocessor.transform(
        X_fold_val
    )

    # --------------------------------------------------------
    # AVISTA Moderate SMOTE (0.5) — FOLD TRAIN only
    # --------------------------------------------------------
    counts = y_fold_train.value_counts()

    majority_class = counts.idxmax()
    minority_class = counts.idxmin()

    majority_count = counts[majority_class]
    minority_count = counts[minority_class]

    target_minority = max(
        minority_count,
        int(np.floor(SMOTE_RATIO * majority_count))
    )

    if target_minority > minority_count:

        smote = SMOTE(
            sampling_strategy={
                minority_class: target_minority
            },
            random_state=RANDOM_SEED
        )

        X_fit, y_fit = smote.fit_resample(
            X_fold_train_processed,
            y_fold_train
        )

        smote_applied = True

    else:
        X_fit = X_fold_train_processed
        y_fit = y_fold_train
        smote_applied = False

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------
    model = LogisticRegression(
        random_state=RANDOM_SEED,
        max_iter=1000
    )

    start = time.perf_counter()

    model.fit(X_fit, y_fit)

    runtime = time.perf_counter() - start

    # --------------------------------------------------------
    # Fold validation
    # --------------------------------------------------------
    y_pred = model.predict(X_fold_val_processed)

    p_index = list(model.classes_).index("p")
    y_prob = model.predict_proba(
        X_fold_val_processed
    )[:, p_index]

    cv_results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, y_pred),
        "Macro-F1": f1_score(
            y_fold_val, y_pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val, y_pred, pos_label="p"
        ),
        "Recall": recall_score(
            y_fold_val, y_pred, pos_label="p"
        ),
        "ROC-AUC": roc_auc_score(
            (y_fold_val == "p").astype(int),
            y_prob
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "SMOTE": smote_applied,
        "Runtime (sec)": runtime
    })

# ------------------------------------------------------------
# Results
# ------------------------------------------------------------
cv_results = pd.DataFrame(cv_results)

display(cv_results)

print("\n=== 5-FOLD CV SUMMARY ===")

metrics = [
    "Accuracy",
    "Macro-F1",
    "Precision",
    "Recall",
    "ROC-AUC"
]

for metric in metrics:
    print(
        f"{metric:10s}: "
        f"{cv_results[metric].mean():.6f} "
        f"± {cv_results[metric].std():.6f}"
    )

,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,SMOTE,Runtime (sec)
0,1,0.980668,0.980644,0.979964,0.979964,0.998754,4548,4548,False,0.022931
1,2,0.984169,0.984148,0.983577,0.983577,0.999027,4549,4549,False,0.018959
2,3,0.983289,0.983278,0.973166,0.992701,0.998321,4549,4549,False,0.021173
3,4,0.986807,0.986796,0.980180,0.992701,0.999510,4549,4549,False,0.021797
4,5,0.981530,0.981516,0.973070,0.989051,0.999080,4549,4549,False,0.019177



=== 5-FOLD CV SUMMARY ===
Accuracy  : 0.983293 ± 0.002404
Macro-F1  : 0.983276 ± 0.002407
Precision : 0.977991 ± 0.004674
Recall    : 0.987599 ± 0.005671
ROC-AUC   : 0.998938 ± 0.000439


In [50]:
# ============================================================
# Cell 10: 5-Fold CV — Random Forest
# ============================================================

from sklearn.ensemble import RandomForestClassifier

rf_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()

    X_fold_val = X_train.iloc[val_idx].copy()
    y_fold_val = y_train.iloc[val_idx].copy()

    # --------------------------------------------------------
    # Preprocessing fitted on fold-training only
    # --------------------------------------------------------
    fold_preprocessor = ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Unknown"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            X_fold_train.columns.tolist()
        )
    ])

    X_fold_train_processed = fold_preprocessor.fit_transform(
        X_fold_train
    )

    X_fold_val_processed = fold_preprocessor.transform(
        X_fold_val
    )

    # --------------------------------------------------------
    # SMOTE Moderate (0.5)
    # --------------------------------------------------------
    counts = y_fold_train.value_counts()

    majority_class = counts.idxmax()
    minority_class = counts.idxmin()

    target_minority = max(
        counts[minority_class],
        int(np.floor(
            SMOTE_RATIO * counts[majority_class]
        ))
    )

    if target_minority > counts[minority_class]:

        smote = SMOTE(
            sampling_strategy={
                minority_class: target_minority
            },
            random_state=RANDOM_SEED
        )

        X_fit, y_fit = smote.fit_resample(
            X_fold_train_processed,
            y_fold_train
        )

        smote_applied = True

    else:
        X_fit = X_fold_train_processed
        y_fit = y_fold_train
        smote_applied = False

    # --------------------------------------------------------
    # Random Forest
    # Match these hyperparameters with AVISTA
    # --------------------------------------------------------
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    start = time.perf_counter()

    model.fit(X_fit, y_fit)

    runtime = time.perf_counter() - start

    y_pred = model.predict(X_fold_val_processed)

    p_index = list(model.classes_).index("p")
    y_prob = model.predict_proba(
        X_fold_val_processed
    )[:, p_index]

    rf_results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(y_fold_val, y_pred),
        "Macro-F1": f1_score(
            y_fold_val, y_pred, average="macro"
        ),
        "Precision": precision_score(
            y_fold_val, y_pred, pos_label="p"
        ),
        "Recall": recall_score(
            y_fold_val, y_pred, pos_label="p"
        ),
        "ROC-AUC": roc_auc_score(
            (y_fold_val == "p").astype(int),
            y_prob
        ),
        "Train N": len(y_fold_train),
        "Effective N": len(y_fit),
        "SMOTE": smote_applied,
        "Runtime (sec)": runtime
    })

rf_results = pd.DataFrame(rf_results)

display(rf_results)

print("\n=== RANDOM FOREST — 5-FOLD CV SUMMARY ===")

for metric in metrics:
    print(
        f"{metric:10s}: "
        f"{rf_results[metric].mean():.6f} "
        f"± {rf_results[metric].std():.6f}"
    )

,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,SMOTE,Runtime (sec)
0,1,1.0,1.0,1.0,1.0,1.0,4548,4548,False,0.217251
1,2,1.0,1.0,1.0,1.0,1.0,4549,4549,False,0.226999
2,3,1.0,1.0,1.0,1.0,1.0,4549,4549,False,0.208910
3,4,1.0,1.0,1.0,1.0,1.0,4549,4549,False,0.218021
4,5,1.0,1.0,1.0,1.0,1.0,4549,4549,False,0.208640



=== RANDOM FOREST — 5-FOLD CV SUMMARY ===
Accuracy  : 1.000000 ± 0.000000
Macro-F1  : 1.000000 ± 0.000000
Precision : 1.000000 ± 0.000000
Recall    : 1.000000 ± 0.000000
ROC-AUC   : 1.000000 ± 0.000000


In [51]:
# ============================================================
# Cell 11: 5-Fold CV — XGBoost
# ============================================================

from xgboost import XGBClassifier

xgb_results = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train, y_train),
    start=1
):
    X_fold_train = X_train.iloc[train_idx].copy()
    y_fold_train = y_train.iloc[train_idx].copy()

    X_fold_val = X_train.iloc[val_idx].copy()
    y_fold_val = y_train.iloc[val_idx].copy()

    # --------------------------------------------------------
    # Preprocessing fitted on fold-training only
    # --------------------------------------------------------
    fold_preprocessor = ColumnTransformer([
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value="Unknown"
                    )
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            X_fold_train.columns.tolist()
        )
    ])

    X_fold_train_processed = fold_preprocessor.fit_transform(
        X_fold_train
    )

    X_fold_val_processed = fold_preprocessor.transform(
        X_fold_val
    )

    # --------------------------------------------------------
    # Convert target for XGBoost
    # edible = 0, poisonous = 1
    # --------------------------------------------------------
    y_fold_train_num = (
        y_fold_train == "p"
    ).astype(int)

    y_fold_val_num = (
        y_fold_val == "p"
    ).astype(int)

    # --------------------------------------------------------
    # SMOTE Moderate (0.5)
    # --------------------------------------------------------
    counts = y_fold_train_num.value_counts()

    majority_class = counts.idxmax()
    minority_class = counts.idxmin()

    target_minority = max(
        counts[minority_class],
        int(np.floor(
            SMOTE_RATIO * counts[majority_class]
        ))
    )

    if target_minority > counts[minority_class]:

        smote = SMOTE(
            sampling_strategy={
                minority_class: target_minority
            },
            random_state=RANDOM_SEED
        )

        X_fit, y_fit = smote.fit_resample(
            X_fold_train_processed,
            y_fold_train_num
        )

        smote_applied = True

    else:
        X_fit = X_fold_train_processed
        y_fit = y_fold_train_num
        smote_applied = False

    # --------------------------------------------------------
    # XGBoost
    # Match these hyperparameters with AVISTA
    # --------------------------------------------------------
    model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=RANDOM_SEED,
        eval_metric="logloss",
        n_jobs=-1
    )

    start = time.perf_counter()

    model.fit(X_fit, y_fit)

    runtime = time.perf_counter() - start

    y_pred = model.predict(X_fold_val_processed)
    y_prob = model.predict_proba(
        X_fold_val_processed
    )[:, 1]

    xgb_results.append({
        "Fold": fold,
        "Accuracy": accuracy_score(
            y_fold_val_num, y_pred
        ),
        "Macro-F1": f1_score(
            y_fold_val_num,
            y_pred,
            average="macro"
        ),
        "Precision": precision_score(
            y_fold_val_num,
            y_pred
        ),
        "Recall": recall_score(
            y_fold_val_num,
            y_pred
        ),
        "ROC-AUC": roc_auc_score(
            y_fold_val_num,
            y_prob
        ),
        "Train N": len(y_fold_train_num),
        "Effective N": len(y_fit),
        "SMOTE": smote_applied,
        "Runtime (sec)": runtime
    })

xgb_results = pd.DataFrame(xgb_results)

display(xgb_results)

print("\n=== XGBOOST — 5-FOLD CV SUMMARY ===")

for metric in metrics:
    print(
        f"{metric:10s}: "
        f"{xgb_results[metric].mean():.6f} "
        f"± {xgb_results[metric].std():.6f}"
    )

,Fold,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Train N,Effective N,SMOTE,Runtime (sec)
0,1,0.999121,0.999120,0.998182,1.000000,1.0,4548,4548,False,0.074049
1,2,1.000000,1.000000,1.000000,1.000000,1.0,4549,4549,False,0.073124
2,3,0.999120,0.999119,1.000000,0.998175,1.0,4549,4549,False,0.123725
3,4,1.000000,1.000000,1.000000,1.000000,1.0,4549,4549,False,0.079386
4,5,1.000000,1.000000,1.000000,1.000000,1.0,4549,4549,False,0.067714



=== XGBOOST — 5-FOLD CV SUMMARY ===
Accuracy  : 0.999648 ± 0.000482
Macro-F1  : 0.999648 ± 0.000482
Precision : 0.999636 ± 0.000813
Recall    : 0.999635 ± 0.000816
ROC-AUC   : 1.000000 ± 0.000000


In [52]:
# ============================================================
# Sanity checks for possible leakage
# ============================================================

print("Target in features:",
      "poisonous" in X_train.columns)

print("Number of features:",
      X_train.shape[1])

print("Duplicate train rows:",
      X_train.duplicated().sum())

print("\nTarget distribution:")
print(y_train.value_counts())

# Check whether any single feature almost perfectly determines target
for col in X_train.columns:
    table = pd.crosstab(X_train[col], y_train)

    purity = table.max(axis=1).sum() / table.values.sum()

    if purity >= 0.95:
        print(f"{col:30s} single-feature purity = {purity:.4f}")

Target in features: False
Number of features: 13
Duplicate train rows: 5144

Target distribution:
poisonous
e    2945
p    2741
Name: count, dtype: int64


In [53]:
# ============================================================
# Cell 12: Final evaluation on untouched 20% test set
# ============================================================

import time
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# ------------------------------------------------------------
# Fit preprocessing on full 70% training set only
# ------------------------------------------------------------
final_preprocessor = ColumnTransformer([
    (
        "categorical",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Unknown"
                )
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                )
            )
        ]),
        X_train.columns.tolist()
    )
])

X_train_final_processed = final_preprocessor.fit_transform(X_train)
X_test_final_processed = final_preprocessor.transform(X_test)

# ------------------------------------------------------------
# SMOTE Moderate (0.5) on training set only
# ------------------------------------------------------------
counts = y_train.value_counts()

majority_class = counts.idxmax()
minority_class = counts.idxmin()

target_minority = max(
    counts[minority_class],
    int(np.floor(0.5 * counts[majority_class]))
)

if target_minority > counts[minority_class]:
    smote = SMOTE(
        sampling_strategy={minority_class: target_minority},
        random_state=42
    )

    X_train_fit, y_train_fit = smote.fit_resample(
        X_train_final_processed,
        y_train
    )
else:
    X_train_fit = X_train_final_processed
    y_train_fit = y_train.copy()

print(f"Original training N  : {len(y_train)}")
print(f"Effective training N : {len(y_train_fit)}")
print(f"Test N               : {len(y_test)}")

# ------------------------------------------------------------
# Models
# ------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        random_state=42,
        max_iter=1000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.3,
        random_state=42,
        eval_metric="logloss",
        n_jobs=-1
    )
}

results = []

for model_name, model in models.items():

    # XGBoost requires numeric labels
    if model_name == "XGBoost":
        y_fit = (y_train_fit == "p").astype(int)
        y_true = (y_test == "p").astype(int)

        start = time.perf_counter()
        model.fit(X_train_fit, y_fit)
        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_final_processed)
        y_prob = model.predict_proba(X_test_final_processed)[:, 1]

        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)

    else:
        y_fit = y_train_fit
        y_true = y_test

        start = time.perf_counter()
        model.fit(X_train_fit, y_fit)
        runtime = time.perf_counter() - start

        y_pred = model.predict(X_test_final_processed)

        p_index = list(model.classes_).index("p")
        y_prob = model.predict_proba(
            X_test_final_processed
        )[:, p_index]

        precision = precision_score(
            y_true, y_pred, pos_label="p"
        )

        recall = recall_score(
            y_true, y_pred, pos_label="p"
        )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro-F1": f1_score(
            y_true, y_pred, average="macro"
        ),
        "Precision": precision,
        "Recall": recall,
        "ROC-AUC": roc_auc_score(
            (y_test == "p").astype(int),
            y_prob
        ),
        "Training N": len(y_train),
        "Effective N": len(y_train_fit),
        "Test N": len(y_test),
        "Runtime (sec)": runtime
    })

test_results = pd.DataFrame(results)

display(
    test_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Runtime (sec)": "{:.4f}"
    })
)

Original training N  : 5686
Effective training N : 5686
Test N               : 1626


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Effective N,Test N,Runtime (sec)
0,Logistic Regression,0.987085,0.987073,0.981084,0.992347,0.999418,5686,5686,1626,0.0230
1,Random Forest,1.000000,1.000000,1.000000,1.000000,1.000000,5686,5686,1626,0.2272
2,XGBoost,1.000000,1.000000,1.000000,1.000000,1.000000,5686,5686,1626,0.0810


In [54]:
# ============================================================
# Cell 13: Feature association with target using Cramér's V
# Training data only
# ============================================================

import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency


def cramers_v(x, y):
    # Treat missing categorical values as Unknown
    x = x.fillna("Unknown")
    y = y.fillna("Unknown")

    confusion_matrix = pd.crosstab(x, y)

    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.values.sum()

    r, k = confusion_matrix.shape

    return np.sqrt(
        (chi2 / n) / min(k - 1, r - 1)
    )


# Calculate association of every feature with target
association_results = []

for feature in X_train.columns:
    score = cramers_v(
        X_train[feature],
        y_train
    )

    association_results.append({
        "Feature": feature,
        "Cramer's V": score
    })

association_df = pd.DataFrame(
    association_results
).sort_values(
    "Cramer's V",
    ascending=False
).reset_index(drop=True)

display(
    association_df.style.format({
        "Cramer's V": "{:.4f}"
    })
)

,Feature,Cramer's V
0,stalk-surface-above-ring,0.5873
1,stalk-surface-below-ring,0.5710
2,gill-size,0.5331
3,bruises,0.4901
4,habitat,0.4305
5,stalk-root,0.4071
6,gill-spacing,0.3540
7,cap-shape,0.2407
8,cap-color,0.2144
9,ring-number,0.2121


In [ ]:
!pip install -U autogluon.tabular

In [56]:
# ============================================================
# AutoGluon — Mushroom
# Same raw 70/10/20 split
# ============================================================

import time
import pandas as pd

from autogluon.tabular import TabularPredictor

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

TARGET = "poisonous"

# ------------------------------------------------------------
# Rebuild raw datasets using the SAME split rows
# No manual preprocessing
# No SMOTE
# ------------------------------------------------------------
ag_train = X_train.copy()
ag_train[TARGET] = y_train.values

ag_val = X_val.copy()
ag_val[TARGET] = y_val.values

ag_test = X_test.copy()
ag_test[TARGET] = y_test.values

print("Train      :", ag_train.shape)
print("Validation :", ag_val.shape)
print("Test       :", ag_test.shape)

# ------------------------------------------------------------
# AutoGluon
# ------------------------------------------------------------
start = time.perf_counter()

predictor = TabularPredictor(
    label=TARGET,
    problem_type="binary",
    eval_metric="accuracy",
    positive_class="p",
    path="AutogluonModels/mushroom"
).fit(
    train_data=ag_train,
    tuning_data=ag_val,
    presets="medium_quality"
)

ag_runtime = time.perf_counter() - start

# ------------------------------------------------------------
# Untouched 20% test set
# ------------------------------------------------------------
X_ag_test = ag_test.drop(columns=TARGET)
y_ag_test = ag_test[TARGET]

y_ag_pred = predictor.predict(X_ag_test)
y_ag_prob = predictor.predict_proba(X_ag_test)

# Probability of poisonous class
if isinstance(y_ag_prob, pd.DataFrame):
    p_prob = y_ag_prob["p"]
else:
    p_prob = y_ag_prob

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
ag_results = pd.DataFrame([{
    "Model": "AutoGluon",
    "Accuracy": accuracy_score(y_ag_test, y_ag_pred),
    "Macro-F1": f1_score(
        y_ag_test, y_ag_pred, average="macro"
    ),
    "Precision": precision_score(
        y_ag_test, y_ag_pred, pos_label="p"
    ),
    "Recall": recall_score(
        y_ag_test, y_ag_pred, pos_label="p"
    ),
    "ROC-AUC": roc_auc_score(
        (y_ag_test == "p").astype(int),
        p_prob
    ),
    "Training N": len(ag_train),
    "Validation N": len(ag_val),
    "Test N": len(ag_test),
    "Runtime (sec)": ag_runtime
}])

display(
    ag_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Runtime (sec)": "{:.2f}"
    })
)

print("\n=== AUTOGLUON LEADERBOARD ===")
display(predictor.leaderboard(ag_test))

Verbosity: 2 (Standard Logging)


Train      : (5686, 14)
Validation : (812, 14)
Test       : (1626, 14)


=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       22.33 GB / 47.93 GB (46.6%)
Disk Space Avail:   267.38 GB / 610.93 GB (43.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
Beginning AutoGluon training ...
AutoGluon will save models to "AutogluonModels\mushroom"
Train Data Rows:    5686
Train Data Columns: 13
Tuning Data Rows:    812
Tuning Data Columns: 13
Label Column:       poisonous
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = p, class 0 = e
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatu

,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Validation N,Test N,Runtime (sec)
0,AutoGluon,1.000000,1.000000,1.000000,1.000000,1.000000,5686,812,1626,67.18



=== AUTOGLUON LEADERBOARD ===


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBM,1.000000,1.0,accuracy,0.034672,0.007589,0.587254,0.034672,0.007589,0.587254,1,True,2
1,LightGBMXT,1.000000,1.0,accuracy,0.034867,0.005030,0.876230,0.034867,0.005030,0.876230,1,True,1
2,NeuralNetFastAI,1.000000,1.0,accuracy,0.040788,0.032744,8.669961,0.040788,0.032744,8.669961,1,True,8
3,WeightedEnsemble_L2,1.000000,1.0,accuracy,0.042702,0.034742,8.770381,0.001915,0.001998,0.100420,2,True,12
4,NeuralNetTorch,1.000000,1.0,accuracy,0.042875,0.016696,22.834746,0.042875,0.016696,22.834746,1,True,10
5,XGBoost,1.000000,1.0,accuracy,0.059475,0.011780,0.412793,0.059475,0.011780,0.412793,1,True,9
6,ExtraTreesEntr,1.000000,1.0,accuracy,0.112134,0.087049,0.943146,0.112134,0.087049,0.943146,1,True,7
7,ExtraTreesGini,1.000000,1.0,accuracy,0.132895,0.091317,0.887033,0.132895,0.091317,0.887033,1,True,6
8,RandomForestGini,1.000000,1.0,accuracy,0.150043,0.075097,0.852632,0.150043,0.075097,0.852632,1,True,3
9,RandomForestEntr,1.000000,1.0,accuracy,0.161383,0.075368,0.827273,0.161383,0.075368,0.827273,1,True,4


In [57]:
# ============================================================
# AutoGluon — Mushroom
# Same 70/10/20 split + 5-fold bagging
# ============================================================

import time
import pandas as pd

from autogluon.tabular import TabularPredictor

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

TARGET = "poisonous"

# ------------------------------------------------------------
# 1. Rebuild RAW train / validation / test datasets
#    Same rows already created earlier
#
#    IMPORTANT:
#    - No one-hot encoding
#    - No manual missing-value imputation
#    - No SMOTE
#    - AutoGluon handles preprocessing internally
# ------------------------------------------------------------

ag_train = X_train.copy()
ag_train[TARGET] = y_train.values

ag_val = X_val.copy()
ag_val[TARGET] = y_val.values

ag_test = X_test.copy()
ag_test[TARGET] = y_test.values

print("=== AUTOGLUON DATA ===")
print(f"Training    : {ag_train.shape}")
print(f"Validation  : {ag_val.shape}")
print(f"Test        : {ag_test.shape}")

print("\nTraining class distribution:")
print(ag_train[TARGET].value_counts())


# ------------------------------------------------------------
# 2. Fit AutoGluon
#
# 5-fold bagging is performed inside the 70% training set.
# The separate 10% validation set is supplied as tuning_data.
# ------------------------------------------------------------

start_time = time.perf_counter()

predictor = TabularPredictor(
    label=TARGET,
    problem_type="binary",
    eval_metric="accuracy",
    positive_class="p",
    path="AutogluonModels/mushroom_5fold"
).fit(
    train_data=ag_train,
    tuning_data=ag_val,

    # AutoGluon platform-level baseline
    presets="medium_quality",

    # 5-fold CV / bagging
    num_bag_folds=5,

    # One complete set of 5 folds
    num_bag_sets=1,

    # Disable multi-layer stacking
    num_stack_levels=0,

    # Required because tuning_data is supplied
    # while bagging is enabled
    use_bag_holdout=True
)

training_runtime = time.perf_counter() - start_time


# ------------------------------------------------------------
# 3. Final evaluation on untouched 20% test set
# ------------------------------------------------------------

X_ag_test = ag_test.drop(columns=TARGET)
y_ag_test = ag_test[TARGET]

prediction_start = time.perf_counter()

y_ag_pred = predictor.predict(X_ag_test)
y_ag_prob = predictor.predict_proba(X_ag_test)

prediction_runtime = time.perf_counter() - prediction_start


# ------------------------------------------------------------
# 4. Probability of poisonous class
# ------------------------------------------------------------

if isinstance(y_ag_prob, pd.DataFrame):

    print("\nProbability columns:")
    print(y_ag_prob.columns.tolist())

    p_prob = y_ag_prob["p"]

else:
    p_prob = y_ag_prob


# ------------------------------------------------------------
# 5. Calculate same metrics as Jupyter / AVISTA
# ------------------------------------------------------------

ag_results = pd.DataFrame([{
    "Platform": "AutoGluon",
    "Accuracy": accuracy_score(
        y_ag_test,
        y_ag_pred
    ),

    "Macro-F1": f1_score(
        y_ag_test,
        y_ag_pred,
        average="macro"
    ),

    "Precision": precision_score(
        y_ag_test,
        y_ag_pred,
        pos_label="p"
    ),

    "Recall": recall_score(
        y_ag_test,
        y_ag_pred,
        pos_label="p"
    ),

    "ROC-AUC": roc_auc_score(
        (y_ag_test == "p").astype(int),
        p_prob
    ),

    "Training N": len(ag_train),

    "Validation N": len(ag_val),

    "Test N": len(ag_test),

    "CV Folds": 5,

    "SMOTE": "No",

    "Training Runtime (sec)": training_runtime,

    "Prediction Runtime (sec)": prediction_runtime
}])


# ------------------------------------------------------------
# 6. Display platform-level test results
# ------------------------------------------------------------

print("\n=== AUTOGLUON — FINAL TEST RESULTS ===")

display(
    ag_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Training Runtime (sec)": "{:.3f}",
        "Prediction Runtime (sec)": "{:.3f}"
    })
)


# ------------------------------------------------------------
# 7. AutoGluon leaderboard on untouched test set
# ------------------------------------------------------------

print("\n=== AUTOGLUON MODEL LEADERBOARD ===")

leaderboard = predictor.leaderboard(
    ag_test,
    extra_info=False
)

display(leaderboard)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       21.67 GB / 47.93 GB (45.2%)
Disk Space Avail:   267.37 GB / 610.93 GB (43.8%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'


=== AUTOGLUON DATA ===
Training    : (5686, 14)
Validation  : (812, 14)
Test        : (1626, 14)

Training class distribution:
poisonous
e    2945
p    2741
Name: count, dtype: int64


Beginning AutoGluon training ...
AutoGluon will save models to "AutogluonModels\mushroom_5fold"
Train Data Rows:    5686
Train Data Columns: 13
Tuning Data Rows:    812
Tuning Data Columns: 13
Label Column:       poisonous
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = p, class 0 = e
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    22156.28 MB
	Train Data (Original)  Memory Usage: 3.99 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting Categ


Probability columns:
['e', 'p']

=== AUTOGLUON — FINAL TEST RESULTS ===


,Platform,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Validation N,Test N,CV Folds,SMOTE,Training Runtime (sec),Prediction Runtime (sec)
0,AutoGluon,1.000000,1.000000,1.000000,1.000000,1.000000,5686,812,1626,5,No,130.210,0.379



=== AUTOGLUON MODEL LEADERBOARD ===


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,LightGBMLarge_BAG_L1,1.000000,1.000000,accuracy,0.086601,0.290745,2.857438,0.086601,0.290745,2.857438,1,True,11
1,CatBoost_BAG_L1,1.000000,1.000000,accuracy,0.126576,0.135795,33.309788,0.126576,0.135795,33.309788,1,True,5
2,ExtraTreesEntr_BAG_L1,1.000000,1.000000,accuracy,0.141111,0.117368,0.850990,0.141111,0.117368,0.850990,1,True,7
3,ExtraTreesGini_BAG_L1,1.000000,1.000000,accuracy,0.141865,0.139737,0.914411,0.141865,0.139737,0.914411,1,True,6
4,RandomForestEntr_BAG_L1,1.000000,1.000000,accuracy,0.145957,0.115728,0.859660,0.145957,0.115728,0.859660,1,True,4
5,NeuralNetTorch_BAG_L1,1.000000,1.000000,accuracy,0.165679,0.259945,28.540956,0.165679,0.259945,28.540956,1,True,10
6,WeightedEnsemble_L2,1.000000,1.000000,accuracy,0.169196,0.261944,28.662885,0.003517,0.001999,0.121929,2,True,12
7,RandomForestGini_BAG_L1,1.000000,1.000000,accuracy,0.169450,0.125205,0.894983,0.169450,0.125205,0.894983,1,True,3
8,NeuralNetFastAI_BAG_L1,1.000000,0.998768,accuracy,0.297696,0.520178,19.018923,0.297696,0.520178,19.018923,1,True,8
9,LightGBM_BAG_L1,0.999385,0.997537,accuracy,0.116507,0.144318,1.884450,0.116507,0.144318,1.884450,1,True,2


In [58]:
# ============================================================
# AutoGluon — Mushroom
# Restricted models: Logistic Regression, Random Forest, XGBoost
# Same raw 70/10/20 split + 5-fold bagging
# ============================================================

import time
import pandas as pd

from autogluon.tabular import TabularPredictor

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

TARGET = "poisonous"


# ============================================================
# 1. Rebuild RAW datasets from the SAME split
# ============================================================
# IMPORTANT:
# - No manual one-hot encoding
# - No manual missing-value imputation
# - No SMOTE
# - AutoGluon handles preprocessing internally
# ============================================================

ag_train = X_train.copy()
ag_train[TARGET] = y_train.values

ag_val = X_val.copy()
ag_val[TARGET] = y_val.values

ag_test = X_test.copy()
ag_test[TARGET] = y_test.values


print("=== AUTOGLUON DATA ===")
print(f"Training    : {ag_train.shape}")
print(f"Validation  : {ag_val.shape}")
print(f"Test        : {ag_test.shape}")

print("\nTraining class distribution:")
print(ag_train[TARGET].value_counts())


# ============================================================
# 2. Fit AutoGluon
# Models restricted to:
# LR  = Logistic Regression
# RF  = Random Forest
# XGB = XGBoost
# ============================================================

start_time = time.perf_counter()

predictor_3models = TabularPredictor(
    label=TARGET,
    problem_type="binary",
    eval_metric="accuracy",
    positive_class="p",
    path="AutogluonModels/mushroom_LR_RF_XGB_5fold"
).fit(

    # Same 70% training rows
    train_data=ag_train,

    # Same 10% validation rows
    tuning_data=ag_val,

    # --------------------------------------------------------
    # Restrict AutoGluon to only these model families
    # --------------------------------------------------------
    hyperparameters={
        "LR": {},
        "RF": {},
        "XGB": {}
    },

    # --------------------------------------------------------
    # 5-fold bagging / CV
    # --------------------------------------------------------
    num_bag_folds=5,
    num_bag_sets=1,

    # No stacking
    num_stack_levels=0,

    # Keep supplied validation set as holdout
    use_bag_holdout=True,

    # Do NOT create WeightedEnsemble_L2
    fit_weighted_ensemble=False,

    # Do not add another final ensemble
    fit_full_last_level_weighted_ensemble=False
)

total_training_runtime = (
    time.perf_counter() - start_time
)


# ============================================================
# 3. Untouched 20% test data
# ============================================================

X_ag_test = ag_test.drop(columns=TARGET)
y_ag_test = ag_test[TARGET]


# ============================================================
# 4. Get AutoGluon leaderboard
# ============================================================

leaderboard = predictor_3models.leaderboard(
    ag_test,
    extra_info=False
)

print("\n=== AUTOGLUON LEADERBOARD ===")
display(leaderboard)


# ============================================================
# 5. Evaluate each trained model separately
# ============================================================

results = []

model_names = leaderboard["model"].tolist()

for model_name in model_names:

    print(f"\nEvaluating: {model_name}")

    start_pred = time.perf_counter()

    # Predictions from this specific model
    y_pred = predictor_3models.predict(
        X_ag_test,
        model=model_name
    )

    y_prob = predictor_3models.predict_proba(
        X_ag_test,
        model=model_name
    )

    prediction_runtime = (
        time.perf_counter() - start_pred
    )

    # Probability of poisonous class
    if isinstance(y_prob, pd.DataFrame):
        p_prob = y_prob["p"]
    else:
        p_prob = y_prob

    results.append({

        "Model": model_name,

        "Accuracy": accuracy_score(
            y_ag_test,
            y_pred
        ),

        "Macro-F1": f1_score(
            y_ag_test,
            y_pred,
            average="macro"
        ),

        "Precision": precision_score(
            y_ag_test,
            y_pred,
            pos_label="p"
        ),

        "Recall": recall_score(
            y_ag_test,
            y_pred,
            pos_label="p"
        ),

        "ROC-AUC": roc_auc_score(
            (y_ag_test == "p").astype(int),
            p_prob
        ),

        "Training N": len(ag_train),

        "Validation N": len(ag_val),

        "Test N": len(ag_test),

        "CV Folds": 5,

        "SMOTE": "No",

        "Prediction Runtime (sec)": prediction_runtime
    })


# ============================================================
# 6. Final comparison table
# ============================================================

ag_3model_results = pd.DataFrame(results)

print(
    "\n=== AUTOGLUON — LR / RF / XGB TEST RESULTS ==="
)

display(
    ag_3model_results.style.format({
        "Accuracy": "{:.6f}",
        "Macro-F1": "{:.6f}",
        "Precision": "{:.6f}",
        "Recall": "{:.6f}",
        "ROC-AUC": "{:.6f}",
        "Prediction Runtime (sec)": "{:.4f}"
    })
)


# ============================================================
# 7. Overall AutoGluon training runtime
# ============================================================

print(
    f"\nTotal AutoGluon training runtime: "
    f"{total_training_runtime:.3f} sec"
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.12.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          12
Pytorch Version:    2.6.0+cu118
CUDA Version:       11.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       21.27 GB / 47.93 GB (44.4%)
Disk Space Avail:   267.35 GB / 610.93 GB (43.8%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by u

=== AUTOGLUON DATA ===
Training    : (5686, 14)
Validation  : (812, 14)
Test        : (1626, 14)

Training class distribution:
poisonous
e    2945
p    2741
Name: count, dtype: int64


Beginning AutoGluon training ...
AutoGluon will save models to "AutogluonModels\mushroom_LR_RF_XGB_5fold"
Train Data Rows:    5686
Train Data Columns: 13
Tuning Data Rows:    812
Tuning Data Columns: 13
Label Column:       poisonous
Problem Type:       binary
Preprocessing data...
Selected class <--> label mapping:  class 1 = p, class 0 = e
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    21757.33 MB
	Train Data (Original)  Memory Usage: 3.99 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special dtypes of the features.
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 4 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fit


=== AUTOGLUON LEADERBOARD ===


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest_BAG_L1,1.000000,1.000000,accuracy,0.127228,0.126339,0.879062,0.127228,0.126339,0.879062,1,True,1
1,XGBoost_BAG_L1,0.999385,1.000000,accuracy,0.131878,0.274225,1.976728,0.131878,0.274225,1.976728,1,True,2
2,LinearModel_BAG_L1,0.984010,0.982759,accuracy,0.085934,0.220294,4.370127,0.085934,0.220294,4.370127,1,True,3



Evaluating: RandomForest_BAG_L1

Evaluating: XGBoost_BAG_L1

Evaluating: LinearModel_BAG_L1

=== AUTOGLUON — LR / RF / XGB TEST RESULTS ===


,Model,Accuracy,Macro-F1,Precision,Recall,ROC-AUC,Training N,Validation N,Test N,CV Folds,SMOTE,Prediction Runtime (sec)
0,RandomForest_BAG_L1,1.000000,1.000000,1.000000,1.000000,1.000000,5686,812,1626,5,No,0.2457
1,XGBoost_BAG_L1,0.999385,0.999384,0.998726,1.000000,1.000000,5686,812,1626,5,No,0.2674
2,LinearModel_BAG_L1,0.984010,0.983998,0.974937,0.992347,0.999294,5686,812,1626,5,No,0.1215



Total AutoGluon training runtime: 16.468 sec
